# 04 - Extract EU Imports and Global Exports

## Purpose
Pull two distinct trade datasets needed for CBAM cost exposure analysis:

1. **EU import trade flows** (Eurostat COMEXT, DS-045409)
   EU27 imports by partner country and CN8 product code, 2020-2024.
   Identifies who exports how much of what into the EU, and at what value.

2. **Global export totals** (UN Comtrade)
   Each CBAM country's total exports to the world (partner = World) for
   CBAM-covered HS6 product codes, 2023-2024.
   Used to compute CBAM cost as a share of total export value -- the
   primary choropleth metric in the project 01 dashboard.

## Outputs
- `data/processed/eu_import_trade_flows.csv`
- `data/processed/comtrade_global_exports.csv`

## Notes
- Comtrade API key stored as environment variable COMTRADE_API_KEY.
  Register at https://uncomtrade.org/docs/how-to-create-an-account/
  then subscribe to comtrade-v1 at https://comtradedeveloper.un.org/
- Comtrade codes are 6-digit HS (not 8-digit CN8). Multiple CN8 codes
  in this pipeline may map to the same HS6 code. This is documented as
  a known limitation: some HS6 codes may capture non-CBAM products
  sharing the same 6-digit prefix.
- 2023 and 2024 are both pulled in a single pass. Fallback resolution
  (prefer 2024, fall back to 2023 where missing) is handled in
  07_clean_transform_data.ipynb.
- Values are returned in USD by Comtrade. EUR conversion is applied
  in the cleaning step using the ECB 2024 annual average rate.
- Mirror data (partner-reported imports) is used as a fallback where
  a country's own export reporting is absent or zero.

In [ ]:
import pandas as pd
import eurostat
import time
from pathlib import Path

# Paths
defaults_path = Path.cwd().parent / "data" / "processed" / "cbam_defaults.csv"
output_path = Path.cwd().parent / "data" / "processed" / "eu_import_trade_flows.csv"

# Load CBAM defaults to get CN code list
df_defaults = pd.read_csv(defaults_path, dtype=str)
df_defaults["cn_clean"] = df_defaults["cn_code"].str.replace(" ", "").str.strip()

print(f"Loaded {len(df_defaults)} rows from cbam_defaults.csv")
print(f"Unique CN codes: {df_defaults['cn_clean'].nunique()}")

Loaded 10671 rows from cbam_defaults.csv
Unique CN codes: 262


## Section 1: CN Code Preparation

Assign each CN code to a CBAM material group based on its numeric prefix. Electricity is excluded from this query. Although electricity is covered by CBAM, it works differently from the other five sectors — instead of a per-tonne default emission value, the CBAM charge for imported electricity is based on how clean or dirty the exporting country's power grid is. That grid data is pulled separately in notebook 03 via Ember. Hydrogen (2804) is separated from other 28xx fertilizer codes.

In [24]:
def assign_material(cn_code):
    cn = str(cn_code).replace(" ", "").strip()
    if cn.startswith("2804"):
        return "hydrogen"
    elif cn.startswith("26") or cn.startswith("72") or cn.startswith("73"):
        return "iron_steel"
    elif cn.startswith("76"):
        return "aluminium"
    elif cn.startswith("28") or cn.startswith("31"):
        return "fertilizers"
    elif cn.startswith("25"):
        return "cement"
    else:
        return None  # electricity and unmapped excluded

df_defaults["material"] = df_defaults["cn_clean"].apply(assign_material)

# Build deduplicated CN code lists per material
materials = ["iron_steel", "aluminium", "cement", "fertilizers", "hydrogen"]
cn_by_material = {}

for material in materials:
    codes = df_defaults[df_defaults["material"] == material]["cn_clean"].unique().tolist()
    cn_by_material[material] = codes
    print(f"{material}: {len(codes)} unique CN codes")

excluded = df_defaults[df_defaults["material"].isna()]["cn_clean"].unique()
print(f"\nExcluded (electricity/unmapped): {len(excluded)} codes")
print(f"Total codes to query: {sum(len(v) for v in cn_by_material.values())}")

iron_steel: 200 unique CN codes
aluminium: 28 unique CN codes
cement: 6 unique CN codes
fertilizers: 27 unique CN codes
hydrogen: 1 unique CN codes

Excluded (electricity/unmapped): 0 codes
Total codes to query: 262


## Section 2: Dataset and Dimension Discovery

Confirmed available COMEXT datasets via `eurostat.get_toc_df(agency='COMEXT')`.
DS-045409 selected as the CN8-level trade dataset. Dimensions and valid
parameter values confirmed below.

In [25]:
DATASET = 'DS-045409'

# Confirm dataset exists and check dimensions
toc = eurostat.get_toc_df(agency='COMEXT')
ds_info = toc[toc['code'] == DATASET][['title', 'code', 'last update of data']]
print("Dataset info:")
print(ds_info.to_string(index=False))

# Confirm dimensions
dims = eurostat.get_pars(DATASET)
print(f"\nDimensions: {dims}")

# Confirm key parameter values
print(f"\nfreq values: {eurostat.get_par_values(DATASET, 'freq')}")
print(f"flow values: {eurostat.get_par_values(DATASET, 'flow')}")
print(f"indicators values: {eurostat.get_par_values(DATASET, 'indicators')}")

reporters = eurostat.get_par_values(DATASET, 'reporter')
eu_codes = [r for r in reporters if 'EU' in str(r)]
print(f"EU reporter codes: {eu_codes}")

Dataset info:
                                 title      code      last update of data
EU trade since 1988 by HS2-4-6 and CN8 DS-045409 2026-04-17T11:00:00+0200

Dimensions: ['freq', 'reporter', 'partner', 'product', 'flow', 'indicators']

freq values: ['A', 'M']
flow values: ['1', '2']
indicators values: ['VALUE_IN_EUROS', 'QUANTITY_IN_100KG', 'SUPPLEMENTARY_QUANTITY']
EU reporter codes: ['EU', 'EU27_2020']


## Section 3: CN Code Format Validation

CBAM defaults contain CN codes at 4-digit (HS4), 6-digit (HS6) and
8-digit (CN8) levels, matching the EU regulation's own classification.

Tested all three lengths against DS-045409 to confirm COMEXT accepts
them as-is. Padded variants (e.g. 72010000, 72024100) are explicitly
rejected by the API with INVALID_QUERY_DIMENSION_VALUE errors.

**Conclusion: CN codes used exactly as stored in cbam_defaults.csv.
No normalization or padding required.**

In [26]:
# Validate CN code formats against API
# Tests: 4-digit, 6-digit, 8-digit originals vs padded equivalents
test_cases = [
    ('7201',     '72010000',  '4-digit HS4'),
    ('720241',   '72024100',  '6-digit HS6'),
    ('72041000', '7204100000','8-digit CN8'),
]

for original, padded, label in test_cases:
    print(f"\n{label}:")
    for code in [original, padded]:
        try:
            df = eurostat.get_data_df(
                DATASET, flags=False,
                filter_pars={
                    'freq': 'A', 'reporter': 'EU27_2020',
                    'partner': 'IN', 'product': code,
                    'flow': '1', 'indicators': 'QUANTITY_IN_100KG'
                }
            )
            status = f"OK - {len(df)} row(s) returned" if df is not None and not df.empty else "empty"
        except Exception as e:
            status = f"REJECTED - {str(e)[:80]}"
        print(f"  {code}: {status}")


4-digit HS4:
  7201: OK - 1 row(s) returned
faultcode: 150
faultstring: INVALID_QUERY_DIMENSION_VALUE: Query is invalid as per its structure's definition. The following values for dimension are not allowed: PRODUCT=72010000.
  72010000: REJECTED - 400 Client Error: Bad Request for url: https://ec.europa.eu/eurostat/api/comext/

6-digit HS6:
  720241: OK - 1 row(s) returned
faultcode: 150
faultstring: INVALID_QUERY_DIMENSION_VALUE: Query is invalid as per its structure's definition. The following values for dimension are not allowed: PRODUCT=72024100.
  72024100: REJECTED - 400 Client Error: Bad Request for url: https://ec.europa.eu/eurostat/api/comext/

8-digit CN8:
  72041000: OK - 1 row(s) returned
faultcode: 150
faultstring: INVALID_QUERY_DIMENSION_VALUE: Query is invalid as per its structure's definition. The following values for dimension are not allowed: PRODUCT=7204100000.
  7204100000: REJECTED - 400 Client Error: Bad Request for url: https://ec.europa.eu/eurostat/api/comext/


## Section 4: Full Extraction

Pulls EU27 import data for all CBAM-covered CN codes by material group.
Both value (euros) and quantity (100kg, converted to tonnes) are extracted.
Years 2020-2024 retained. Partial 2025 data excluded.

Output is long format with one row per country-product-year-indicator.

In [30]:
TARGET_YEARS = [str(y) for y in range(2020, 2025)]
INDICATORS = ['VALUE_IN_EUROS', 'QUANTITY_IN_100KG']

def fetch_material(material, cn_codes):
    """Fetch EU27 import data for a list of CN codes."""
    all_dfs = []
    errors = []

    for i, cn in enumerate(cn_codes):
        try:
            df = eurostat.get_data_df(
                DATASET, flags=False,
                filter_pars={
                    'freq': 'A',
                    'reporter': 'EU27_2020',
                    'product': cn,
                    'flow': '1',
                    'indicators': INDICATORS
                }
            )

            if df is None or df.empty:
                continue

            # Keep only target years
            id_col = 'indicators\\TIME_PERIOD'
            id_vars = ['freq', 'reporter', 'partner', 'product', 'flow', id_col]
            year_cols = [c for c in TARGET_YEARS if c in df.columns]
            df = df[id_vars + year_cols]

            # Melt to long format
            df_long = df.melt(
                id_vars=id_vars,
                var_name='year',
                value_name='value'
            )
            df_long = df_long.rename(columns={id_col: 'indicator'})
            df_long['material'] = material
            df_long = df_long[df_long['value'].notna()]
            all_dfs.append(df_long)

        except Exception as e:
            errors.append((cn, str(e)))

        time.sleep(0.3)

        if (i + 1) % 20 == 0:
            print(f"  {i + 1}/{len(cn_codes)} codes processed...")

    if errors:
        print(f"  Errors on {len(errors)} codes: {errors[:3]}")

    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()


# Run extraction by material group
all_results = []

for material in materials:
    codes = cn_by_material[material]
    print(f"\nFetching {material} ({len(codes)} codes)...")
    df_mat = fetch_material(material, codes)
    if not df_mat.empty:
        all_results.append(df_mat)
        print(f"  Done: {len(df_mat)} rows")
    else:
        print(f"  No data returned")

# Combine all materials
df_all = pd.concat(all_results, ignore_index=True)

# Convert quantity from 100kg to tonnes
df_all.loc[df_all['indicator'] == 'QUANTITY_IN_100KG', 'value'] = \
    df_all.loc[df_all['indicator'] == 'QUANTITY_IN_100KG', 'value'] / 10
df_all['indicator'] = df_all['indicator'].replace(
    'QUANTITY_IN_100KG', 'QUANTITY_IN_TONNES'
)

print(f"\nTotal rows extracted: {len(df_all)}")
print(f"Materials: {df_all['material'].unique()}")
print(f"Years: {sorted(df_all['year'].unique())}")
print(f"Partner countries: {df_all['partner'].nunique()}")
print(f"\nSample:")
print(df_all.head(10).to_string())


Fetching iron_steel (200 codes)...
  20/200 codes processed...
  40/200 codes processed...
  60/200 codes processed...
  80/200 codes processed...
  100/200 codes processed...
  120/200 codes processed...
  140/200 codes processed...
  160/200 codes processed...
  180/200 codes processed...
  200/200 codes processed...
  Done: 122820 rows

Fetching aluminium (28 codes)...
  20/28 codes processed...
  Done: 25502 rows

Fetching cement (6 codes)...
  Done: 4108 rows

Fetching fertilizers (27 codes)...
  20/27 codes processed...
  Done: 12286 rows

Fetching hydrogen (1 codes)...
  Done: 466 rows

Total rows extracted: 165182
Materials: <StringArray>
['iron_steel', 'aluminium', 'cement', 'fertilizers', 'hydrogen']
Length: 5, dtype: str
Years: ['2020', '2021', '2022', '2023', '2024']
Partner countries: 244

Sample:
  freq   reporter partner   product flow       indicator  year        value    material
0    A  EU27_2020      AR  26011200    1  VALUE_IN_EUROS  2020         30.0  iron_steel
1

In [31]:
# Save to processed
df_all.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Final shape: {df_all.shape}")

Saved to: /Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/comext_trade_flows.csv
Final shape: (165182, 9)


# Section 2: UN Comtrade Global Export Extraction

In [ ]:
# Pulls total export values and quantities to the world (partner = 0) for all
# 119 CBAM countries and all CBAM-relevant HS6 product codes for years 2023
# and 2024. Both years are extracted in a single pass; fallback resolution
# (prefer 2024, use 2023 if 2024 is absent) is deferred to the cleaning step.
#
# Batching strategy:
#   The Comtrade API accepts comma-separated cmdCode values (max 20 per call)
#   and comma-separated reporterCode values (no hard cap). With ~82 distinct
#   HS6 codes across 5 sectors, we need ceil(82/20) = ~5 calls total.
#   All 119 reporter codes are sent in each call, producing at most
#   ~4,760 rows per response -- well within the 100,000-record limit.
#   This keeps total API calls far below the 500/day free-tier limit.

import os
import time
import math
import pandas as pd
import comtradeapicall
from pathlib import Path

# Paths
crosswalk_path = Path("../data/clean/country_crosswalk.csv")
output_path    = Path("../data/processed/comtrade_global_exports.csv")

# Load the crosswalk to get the 119 CBAM countries and their ISO3 codes.
# Comtrade uses numeric reporter codes; the package's convertCountryIso3ToCode
# function handles the ISO3 -> numeric conversion for us.
crosswalk = pd.read_csv(crosswalk_path, dtype=str)
cbam_countries = crosswalk[crosswalk["name_cbam_defaults"].notna()].copy()

print(f"CBAM countries to query: {len(cbam_countries)}")
print(f"Namibia included (iso2=None, iso3=NAM): {'NAM' in cbam_countries['iso3'].values}")
print(f"\nSample:\n{cbam_countries[['country', 'iso2', 'iso3']].head(8).to_string(index=False)}")

CBAM countries to query: 119
Namibia included (iso2=None, iso3=NAM): True

Sample:
   country iso2 iso3
   Albania   AL  ALB
   Algeria   DZ  DZA
    Angola   AO  AGO
 Argentina   AR  ARG
   Armenia   AM  ARM
 Australia   AU  AUS
Azerbaijan   AZ  AZE
   Bahrain   BH  BHR


In [3]:
# ── Derive the HS6 code list from the CBAM defaults ──────────────────────────
#
# CBAM CN8 codes are truncated to 6 digits for Comtrade compatibility.
# Multiple CN8 codes sharing a prefix collapse to the same HS6 code.
# Electricity (27xx) is excluded -- it is handled separately via Ember.
# Deduplication is applied after truncation.
#
# Known limitation: some HS6 codes capture products beyond CBAM scope that
# share the 6-digit prefix. This is unavoidable at the HS6 level and is
# documented in the output metadata.

defaults_path = Path("../data/processed/cbam_defaults.csv")
df_defaults = pd.read_csv(defaults_path, dtype=str)
df_defaults["cn_clean"] = df_defaults["cn_code"].str.replace(" ", "").str.strip()

def assign_sector(cn_code):
    """
    Assign a CBAM sector label to a CN code based on its numeric prefix.
    Returns None for electricity and any unmapped codes, which are excluded
    from the Comtrade query.
    """
    cn = str(cn_code).replace(" ", "").strip()
    if cn.startswith("2804"):
        return "hydrogen"
    elif cn.startswith("26") or cn.startswith("72") or cn.startswith("73"):
        return "iron_steel"
    elif cn.startswith("76"):
        return "aluminium"
    elif cn.startswith("28") or cn.startswith("31"):
        return "fertilizers"
    elif cn.startswith("25") or cn.startswith("68"):
        return "cement"
    else:
        return None  # electricity (27xx) and unmapped codes excluded

df_defaults["sector"] = df_defaults["cn_clean"].apply(assign_sector)

# Truncate CN8 codes to 6 digits to produce HS6 codes for Comtrade.
# Shorter codes (4-digit HS4) are left as-is -- they are valid Comtrade
# cmdCode values and cover the correct product scope.
df_defaults["hs6"] = df_defaults["cn_clean"].str[:6]

# Build the deduplicated HS6 list per sector, excluding electricity
hs6_by_sector = (
    df_defaults[df_defaults["sector"].notna()]
    .groupby("sector")["hs6"]
    .apply(lambda x: sorted(x.unique().tolist()))
    .to_dict()
)

all_hs6_codes = sorted(set(
    code for codes in hs6_by_sector.values() for code in codes
))

print(f"Distinct HS6 codes to query: {len(all_hs6_codes)}")
print(f"Code batches needed (max 20 per call): {math.ceil(len(all_hs6_codes) / 20)}")
print(f"\nBreakdown by sector:")
for sector, codes in hs6_by_sector.items():
    print(f"  {sector}: {len(codes)} HS6 codes")
print(f"\nFull HS6 code list:\n{all_hs6_codes}")

Distinct HS6 codes to query: 196
Code batches needed (max 20 per call): 10

Breakdown by sector:
  aluminium: 21 HS6 codes
  cement: 6 HS6 codes
  fertilizers: 20 HS6 codes
  hydrogen: 1 HS6 codes
  iron_steel: 148 HS6 codes

Full HS6 code list:
['250700', '252310', '252321', '252329', '252330', '252390', '260112', '280410', '280800', '281410', '281420', '283421', '310210', '310221', '310229', '310230', '310240', '310250', '310260', '310280', '310290', '310510', '310520', '310530', '310540', '310551', '310559', '310590', '7201', '720211', '720219', '720241', '720249', '720260', '7203', '7205', '720610', '720690', '720711', '720712', '720719', '720720', '7208', '7209', '7210', '721113', '721114', '721119', '721123', '721129', '721190', '7212', '7213', '721410', '721420', '721430', '721491', '721499', '7215', '7216', '721710', '721720', '721730', '721790', '721810', '721891', '721899', '721911', '721912', '721913', '721914', '721921', '721922', '721923', '721924', '721931', '721932', '72

In [6]:
# Load environment variables from the .env file in the project root.
# This makes COMTRADE_API_KEY available to os.environ.get() in subsequent cells.
# The .env file is excluded from version control via .gitignore.
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path("../.env"))
print("Environment variables loaded.")

Environment variables loaded.


In [8]:
# ── Convert ISO3 codes to Comtrade numeric reporter codes ─────────────────────
#
# Comtrade's API uses numeric reporter codes internally. The package provides
# convertCountryIso3ToCode() which fetches the mapping from the UN reference
# file and returns a comma-separated string of numeric codes.
#
# Namibia (NAM) has no ISO2 code but is present in Comtrade's reporter list
# under ISO3=NAM. It is included here and its iso2=None is handled downstream
# in the cleaning step.

api_key = os.environ.get("COMTRADE_API_KEY")
if not api_key:
    raise EnvironmentError(
        "COMTRADE_API_KEY environment variable is not set.\n"
        "Register at https://uncomtrade.org/docs/how-to-create-an-account/ "
        "and subscribe to comtrade-v1 at https://comtradedeveloper.un.org/"
    )

# Build the ISO3 string for the converter (comma-separated)
iso3_list = cbam_countries["iso3"].dropna().tolist()
iso3_string = ",".join(iso3_list)

# Fetch numeric reporter codes from the UN reference endpoint
print("Fetching Comtrade reporter codes from UN reference file...")
reporter_code_string = comtradeapicall.convertCountryIso3ToCode(iso3_string)
reporter_codes = [c.strip() for c in reporter_code_string.split(",") if c.strip()]

print(f"ISO3 codes submitted:              {len(iso3_list)}")
print(f"Numeric reporter codes returned:   {len(reporter_codes)}")

# Flag any ISO3 codes that did not resolve (absent from UN reference file)
resolved_count = len(reporter_codes)
if resolved_count < len(iso3_list):
    print(f"\nWARNING: {len(iso3_list) - resolved_count} ISO3 code(s) did not resolve.")
    print("These countries will be absent from the Comtrade results.")
    print("Check the crosswalk for any non-standard ISO3 codes (e.g. XKX for Kosovo).")
else:
    print("\nAll ISO3 codes resolved successfully.")

print(f"\nReporter code string (first 120 chars): {reporter_code_string[:120]}...")

Fetching Comtrade reporter codes from UN reference file...
ISO3 codes submitted:              119
Numeric reporter codes returned:   128

All ISO3 codes resolved successfully.

Reporter code string (first 120 chars): 8,12,24,886,32,51,36,31,48,50,112,204,68,70,76,96,116,120,124,152,156,344,170,178,188,384,192,531,408,180,214,588,218,81...


In [ ]:
# ── Define the extraction function for a single code batch ───────────────────
#
# Each call fetches all 119 reporters, partner=0 (World), flow=X (exports),
# for one batch of up to 20 HS6 codes, for period "2023,2024".
#
# The function returns a raw DataFrame with one row per
# (reporter, cmdCode, period) combination.
#
# Retry logic: up to 3 attempts with exponential backoff on failure.
# On persistent failure the batch is logged and skipped rather than
# crashing the full run -- partial results are still saved.

MAX_RETRIES    = 3
RETRY_DELAY_S  = 5   # seconds before first retry; doubled on each attempt

def fetch_comtrade_batch(
    api_key,
    reporter_code_string,
    hs6_codes,
    period="2023,2024",
    flow="X",
    partner="0",
):
    """
    Fetch Comtrade annual goods trade data for a batch of HS6 codes.

    Parameters
    ----------
    api_key             : str   Comtrade subscription key.
    reporter_code_string: str   Comma-separated numeric reporter codes.
    hs6_codes           : list  Up to 20 HS6 code strings.
    period              : str   Comma-separated year(s), e.g. "2023,2024".
    flow                : str   "X" for exports, "M" for imports.
    partner             : str   "0" = World aggregate.

    Returns
    -------
    pd.DataFrame or None if all retries failed.
    """
    cmd_string = ",".join(hs6_codes)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            df = comtradeapicall.getFinalData(
                subscription_key = api_key,
                typeCode         = "C",       # commodities (goods)
                freqCode         = "A",       # annual
                clCode           = "HS",      # Harmonized System
                period           = period,
                reporterCode     = reporter_code_string,
                cmdCode          = cmd_string,
                flowCode         = flow,
                partnerCode      = partner,
                partner2Code     = "0",       # no secondary partner dimension
                customsCode      = "C00",     # standard customs (all procedures)
                motCode          = "0",       # all modes of transport
            )

            if df is None or df.empty:
                return pd.DataFrame()

            return df

        except Exception as exc:
            wait = RETRY_DELAY_S * (2 ** (attempt - 1))
            print(f"  Attempt {attempt}/{MAX_RETRIES} failed: {exc}")
            if attempt < MAX_RETRIES:
                print(f"  Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  All retries exhausted for batch: {hs6_codes[:3]}...")
                return None

print("Extraction function defined.")
print(f"Settings: flow=X (exports), partner=0 (World), period=2023,2024")
print(f"Retry policy: {MAX_RETRIES} attempts, exponential backoff from {RETRY_DELAY_S}s")

Extraction function defined.
Settings: flow=X (exports), partner=0 (World), period=2023,2024
Retry policy: 3 attempts, exponential backoff from 5s


In [ ]:
# ── Primary extraction: reporter-reported export data ────────────────────────
#
# Iterates over code batches of size 20. Each call covers all 119 reporters
# simultaneously. Both 2023 and 2024 are requested in a single period string
# to minimize API calls.
#
# Progress is printed per batch. Failed batches are collected in
# failed_batches for the retry/mirror pass below.

BATCH_SIZE = 20

# Split the full HS6 code list into batches of at most BATCH_SIZE
code_batches = [
    all_hs6_codes[i : i + BATCH_SIZE]
    for i in range(0, len(all_hs6_codes), BATCH_SIZE)
]

print(f"Starting primary extraction pass (reporter-reported)")
print(f"Total code batches: {len(code_batches)}")
print(f"Codes per batch: {BATCH_SIZE} (last batch may be smaller)\n")

primary_results = []
failed_batches  = []

for batch_num, batch in enumerate(code_batches, start=1):
    print(f"Batch {batch_num}/{len(code_batches)}: codes {batch[0]}...{batch[-1]}")
    df_batch = fetch_comtrade_batch(
        api_key              = api_key,
        reporter_code_string = reporter_code_string,
        hs6_codes            = batch,
    )

    if df_batch is None:
        print(f"  FAILED -- added to retry list")
        failed_batches.append(batch)
    elif df_batch.empty:
        print(f"  No data returned")
    else:
        primary_results.append(df_batch)
        print(f"  OK -- {len(df_batch):,} rows")

    # Polite delay between calls to avoid rate-limit responses
    time.sleep(1.5)

# Combine all successful primary batches
if primary_results:
    df_primary = pd.concat(primary_results, ignore_index=True)
else:
    df_primary = pd.DataFrame()

print(f"\nPrimary pass complete.")
print(f"  Successful batches: {len(primary_results)} / {len(code_batches)}")
print(f"  Failed batches:     {len(failed_batches)}")
print(f"  Total rows:         {len(df_primary):,}")
if not df_primary.empty:
    print(f"  Columns: {df_primary.columns.tolist()}")

Starting primary extraction pass (reporter-reported)
Total code batches: 10
Codes per batch: 20 (last batch may be smaller)

Batch 1/10: codes 250700...310280
  OK -- 1,859 rows
Batch 2/10: codes 310290...720712
  OK -- 1,595 rows
Batch 3/10: codes 720719...7216
  OK -- 2,243 rows
Batch 4/10: codes 721710...721935
  OK -- 1,562 rows
Batch 5/10: codes 721990...722591
  OK -- 1,636 rows
Batch 6/10: codes 722592...7302
  OK -- 1,668 rows
Batch 7/10: codes 730300...730630
  OK -- 2,350 rows
Batch 8/10: codes 730640...731811
  OK -- 2,973 rows
Batch 9/10: codes 731812...760429
  OK -- 2,990 rows
Batch 10/10: codes 7605...761699
  OK -- 2,307 rows

Primary pass complete.
  Successful batches: 10 / 10
  Failed batches:     0
  Total rows:         21,183
  Columns: ['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period', 'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc', 'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code', 'partner2ISO', 'partner2Desc

In [22]:
# ── Normalise column names, recover iso3 from numeric reporter code, and save ─
#
# The Comtrade API does not reliably populate reporterISO in responses.
# reporterCode (numeric) is always present and is used to recover iso3
# by joining against the UN country area code reference file.
#
# Reference file: https://comtradeapi.un.org/files/v1/app/reference/country_area_code_iso.json
# Fetched fresh at runtime to ensure the mapping is current.

import urllib3, json
from pandas import json_normalize

# Fetch the numeric code -> ISO3 reference table from the UN
http = urllib3.PoolManager()
resp = http.request(
    "GET",
    "https://comtradeapi.un.org/files/v1/app/reference/country_area_code_iso.json",
    timeout=30,
)
ref = json_normalize(json.loads(resp.data)["results"])
ref["country_area_code"] = ref["country_area_code"].astype(str)
print(f"Reporter reference table: {len(ref)} rows")

# Column rename map -- reporterISO excluded (unreliable), reporterCode
# kept so we can join to the reference table to recover iso3
COLUMN_MAP = {
    "reporterCode"       : "reporter_code",
    "reporterDesc"       : "reporter_name_raw",
    "cmdCode"            : "hs6_code",
    "cmdDesc"            : "hs6_desc",
    "period"             : "year",
    "primaryValue"       : "export_value_usd",
    "netWgt"             : "export_netweight_kg",
    "qty"                : "export_qty",
    "qtyUnitAbbr"        : "qty_unit",
}

cols_present = {k: v for k, v in COLUMN_MAP.items() if k in df_primary.columns}
missing      = [k for k in COLUMN_MAP if k not in df_primary.columns]
if missing:
    print(f"NOTE: expected columns absent from API response: {missing}")

df_out = df_primary.rename(columns=cols_present)
df_out = df_out[list(cols_present.values())].copy()

# Recover iso3 by joining on numeric reporter_code
df_out["reporter_code"] = df_out["reporter_code"].astype(str)
df_out = df_out.merge(ref, left_on="reporter_code", right_on="country_area_code", how="left")
df_out = df_out.drop(columns=["country_area_code", "reporter_code"])

# Confirm iso3 recovery
null_iso3 = df_out["iso3"].isnull().sum()
print(f"\nRows with null iso3 after recovery: {null_iso3}")
if null_iso3 > 0:
    print("These are Comtrade aggregate/unspecified reporter codes with no ISO3 equivalent.")
    print("They will be dropped in the cleaning step.")
    print(df_out[df_out["iso3"].isnull()]["reporter_name_raw"].value_counts())

# Ensure year is integer, hs6_code is zero-padded string
df_out["year"]     = pd.to_numeric(df_out["year"], errors="coerce").astype("Int64")
df_out["hs6_code"] = df_out["hs6_code"].astype(str).str.strip()

print(f"\nFinal shape: {df_out.shape}")
print(f"Columns:     {df_out.columns.tolist()}")
print(f"Years:       {sorted(df_out['year'].dropna().unique())}")
print(f"\nSample:\n{df_out.head(6).to_string(index=False)}")

df_out.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

Reporter reference table: 312 rows

Rows with null iso3 after recovery: 0

Final shape: (21183, 9)
Columns:     ['reporter_name_raw', 'hs6_code', 'hs6_desc', 'year', 'export_value_usd', 'export_netweight_kg', 'export_qty', 'qty_unit', 'iso3']
Years:       [np.int64(2023), np.int64(2024)]

Sample:
reporter_name_raw hs6_code hs6_desc  year  export_value_usd  export_netweight_kg  export_qty qty_unit iso3
             None   252321     None  2023        107927.979             590270.0    590270.0     None  ALB
             None   252329     None  2023      60214695.760          583570265.0 583570265.0     None  ALB
             None   310210     None  2023       1346655.686            2288480.0   2288480.0     None  ALB
             None   310230     None  2023         96213.189             179300.0    179300.0     None  ALB
             None   250700     None  2023           656.914                100.0       100.0     None  AGO
             None   252310     None  2023      16579319.508 